# Stage 07 · Multi-day scenario evaluation

This notebook delegates tested logic to `src/mfar_core.py` and writes auditable artifacts.

In [ ]:
# Shared paths and deterministic stage initialization
from pathlib import Path
from datetime import datetime, timezone
import os, sys
import pandas as pd

root = Path(os.environ.get("MFAR_CODE_ROOT", Path.cwd()))
if not (root / "src" / "mfar_paths.py").is_file():
    root = Path.cwd().parent
if not (root / "src" / "mfar_paths.py").is_file():
    raise FileNotFoundError("Run the notebook from the repository or set MFAR_CODE_ROOT")
sys.path.insert(0, str(root.resolve()))

from src.mfar_paths import *
from src.mfar_core import *
STARTED_AT = datetime.now(timezone.utc)


In [ ]:
NOTEBOOK_NAME = "07_Candidate_Action.ipynb"
validate_writable_directory(STAGE_DIRS[7], NOTEBOOK_NAME, 7)


In [ ]:
queue = validate_csv_input(STAGE_04_DIR / "04_daily_port_queue_forecast.csv",
    ["simulation_time", "port_id", "queue_ce", "queue_ratio"], NOTEBOOK_NAME, 7)
queue["simulation_time"] = pd.to_datetime(queue["simulation_time"], errors="coerce")
events = validate_csv_input(STAGE_04_DIR / "04_daily_event_log.csv",
    ["simulation_time", "origin", "served_ce", "mmsi"], NOTEBOOK_NAME, 7)
events["simulation_time"] = pd.to_datetime(events["simulation_time"], errors="coerce")
evaluated = validate_csv_input(STAGE_06_DIR / "06_rule_evaluation.csv",
    ["simulation_time", "origin", "mmsi", "operational_phase", "selected_action",
     "selected_rule_strength", "dominant_rule"], NOTEBOOK_NAME, 7)
evaluated["simulation_time"] = pd.to_datetime(evaluated["simulation_time"], errors="coerce")
rates = pd.read_csv(VEHICLE_ARRIVAL_PATH)
profiles = pd.read_csv(CONFIG_DIR / "vessel_profiles.csv")
sim, accepted, effects, daily, overall = run_stage7(
    queue, events, evaluated, rates, profiles, STAGE_07_DIR, CONFIG_DIR)
display(overall); display(daily.head(12))
print("Stage 07 is scenario evaluation, not empirical intervention validation.")


In [ ]:
NOTEBOOK_NAME = "07_Candidate_Action.ipynb"
stage_dir = STAGE_DIRS[7]
files = sorted(stage_dir.glob("07_*"))
write_execution_metadata(7, NOTEBOOK_NAME, STARTED_AT,
    [CONFIG_DIR / "pipeline_parameters.csv"], {}, {"primary": len(locals().get("state", locals().get("forecast", locals().get("sim", []))))}, files)
